In [ ]:
get_ipython().system("pip install sentence-transformers faiss-cpu numpy pandas rank_bm25 openai-whisper -q")
get_ipython().system("apt-get install -y ffmpeg -q")
print("✅ All packages installed")

Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.
✅ All packages installed


In [ ]:
import os
import re
import numpy as np
import pandas as pd
import faiss
import whisper
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

print("✅ All imports successful")

✅ All imports successful


In [ ]:
CSV_PATH        = "/content/finalized_data (1) (1).csv"     # your CSV file
EMBEDDINGS_PATH = "scheme_embeddings.npy"       # saved embeddings
INDEX_PATH      = "scheme_faiss.index"          # saved FAISS index

# ── Models ───────────────────────────────────────────────────────
EMBED_MODEL   = "paraphrase-multilingual-MiniLM-L12-v2"
WHISPER_MODEL = "medium"   # medium = best for Indian languages

# ── Search config ────────────────────────────────────────────────
TOP_K          = 5
SEMANTIC_WEIGHT = 0.7   # 70% semantic score
BM25_WEIGHT     = 0.3   # 30% keyword score
# Together = hybrid score. Change these to tune retrieval quality.

print(f"✅ Config loaded")
print(f"   Semantic weight : {SEMANTIC_WEIGHT}")
print(f"   BM25 weight     : {BM25_WEIGHT}")

✅ Config loaded
   Semantic weight : 0.7
   BM25 weight     : 0.3


In [ ]:
df = pd.read_csv(CSV_PATH, on_bad_lines="skip")
df.reset_index(drop=True, inplace=True)

print(f"✅ Loaded {len(df):,} rows, {len(df.columns)} columns")
print(f"   Columns: {df.columns.tolist()}")

✅ Loaded 3,400 rows, 15 columns
   Columns: ['scheme_name', 'tags', 'tags_hi', 'tags_kn', 'details', 'details_hi', 'slug', 'benefits', 'eligibility', 'application', 'documents', 'level', 'schemeCategory', 'scheme_name_hindi', 'scheme_name_kannada']


In [ ]:
def safe(val):
    """Convert cell to string, empty string for NaN."""
    return "" if pd.isna(val) else str(val).strip()

def build_search_text(row):
    """
    Combines English + Hindi + Kannada fields into one string.
    A query in ANY language will match against this.
    """
    parts = [
        # Scheme names in all 3 languages
        safe(row.get("scheme_name",          "")),
        safe(row.get("scheme_name_hindi",    "")),
        safe(row.get("scheme_name_kannada",  "")),
        # Tags in all 3 languages
        safe(row.get("tags",     "")),
        safe(row.get("tags_hi",  "")),
        safe(row.get("tags_kn",  "")),
        # Category + level
        safe(row.get("schemeCategory", "")),
        safe(row.get("level",          "")),
        # Details — truncated to keep token count safe
        safe(row.get("details",    ""))[:400],
        safe(row.get("details_hi", ""))[:200],
        # Eligibility snippet — very useful for queries like "who can apply"
        safe(row.get("eligibility", ""))[:200],
        # Benefits snippet
        safe(row.get("benefits", ""))[:200],
    ]
    return " | ".join(p for p in parts if p)

df["search_text"] = df.apply(build_search_text, axis=1)

print("✅ search_text column built")
print(f"\nSample (row 0):\n{df['search_text'].iloc[0][:300]}...")

✅ search_text column built

Sample (row 0):
"Immediate Relief Assistance" under "Welfare and Relief for Fishermen During Lean Seasons and Natural Calamities Scheme" | इमीडियेट रिलीफ असिस्टेंस अंडर वेलफेयर एंड रिलीफ फॉर फिशरमैन दूरिंग लेअन सेअसोंस एंड नेचुरल कालमीटीएस स्कीम | ಇಮ್ಮೆಡೈತೆ ರಿಲೀಫ್ ಅಸ್ಸಿಸ್ಟಂಸ್ ಅಂಡರ್ ವೆಲ್ಫೇರ್ ಅಂಡ್ ರಿಲೀಫ್ ಫಾರ್ ಫಿಶರ್ಮ್...


In [ ]:
print(f"Loading sentence-transformer: {EMBED_MODEL}")
embed_model = SentenceTransformer(EMBED_MODEL)
print("✅ Embedding model ready")

Loading sentence-transformer: paraphrase-multilingual-MiniLM-L12-v2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model ready


In [ ]:
if os.path.exists(EMBEDDINGS_PATH):
    print(f"Loading saved embeddings from {EMBEDDINGS_PATH}...")
    embeddings = np.load(EMBEDDINGS_PATH)
else:
    print(f"Generating embeddings for {len(df):,} rows...")
    print("  (~2-5 mins on CPU, ~30 sec on GPU)")
    embeddings = embed_model.encode(
        df["search_text"].tolist(),
        batch_size          = 64,
        show_progress_bar   = True,
        convert_to_numpy    = True,
        normalize_embeddings= True,  # L2 normalize for cosine via dot product
    )
    np.save(EMBEDDINGS_PATH, embeddings)
    print(f"✅ Embeddings saved — shape: {embeddings.shape}")

print(f"✅ Embeddings ready — shape: {embeddings.shape}")

# ── FAISS index ──────────────────────────────────────────────────
if os.path.exists(INDEX_PATH):
    print(f"Loading saved FAISS index from {INDEX_PATH}...")
    index = faiss.read_index(INDEX_PATH)
else:
    print("Building FAISS index...")
    dim   = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings.astype(np.float32))
    faiss.write_index(index, INDEX_PATH)
    print(f"✅ FAISS index saved — {index.ntotal:,} vectors")

print(f"✅ FAISS index ready — {index.ntotal:,} vectors")

Generating embeddings for 3,400 rows...
  (~2-5 mins on CPU, ~30 sec on GPU)


Batches:   0%|          | 0/54 [00:00<?, ?it/s]

✅ Embeddings saved — shape: (3400, 384)
✅ Embeddings ready — shape: (3400, 384)
Building FAISS index...
✅ FAISS index saved — 3,400 vectors
✅ FAISS index ready — 3,400 vectors


In [ ]:
def tokenize(text):
    """Simple whitespace + punctuation tokenizer."""
    return re.findall(r"\w+", str(text).lower())

print("Building BM25 index...")
tokenized_corpus = [tokenize(t) for t in df["search_text"]]
bm25 = BM25Okapi(tokenized_corpus)
print(f"✅ BM25 index ready — {len(tokenized_corpus):,} documents")

Building BM25 index...
✅ BM25 index ready — 3,400 documents


In [ ]:
def hybrid_search(query: str, top_k: int = TOP_K) -> pd.DataFrame:
    query_clean = query.strip()
    seen_indices = set()
    results = []

    # ── Part 1: Fuzzy name match ──────────────────────────────────────────────
    fuzzy_df = fuzzy_name_match(query_clean, top_k=top_k)

    for _, row in fuzzy_df.iterrows():
        idx = row["idx"]
        seen_indices.add(idx)
        results.append({
            "rank":                 None,
            "similarity":           round(row["fuzzy_score"] / 100, 4),
            "match_type":           f"fuzzy ({row['fuzzy_score']}%)",
            "scheme_name":          row["scheme_name"],
            "scheme_name_hindi":    row["scheme_name_hindi"],
            "scheme_name_kannada":  row["scheme_name_kannada"],
            "tags":                 row["tags"],
            "level":                row["level"],
            "schemeCategory":       row["schemeCategory"],
            "slug":                 row["slug"],                  # ← added
            "details_snippet":      row["details_snippet"],
        })

    # ── Part 2: Semantic FAISS search ─────────────────────────────────────────
    query_embedding = embed_model.encode( # Changed 'model.encode' to 'embed_model.encode'
        [query_clean],
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype(np.float32)

    scores, indices = index.search(query_embedding, top_k * 2)

    for score, idx in zip(scores[0], indices[0]):
        if idx in seen_indices:
            continue
        seen_indices.add(idx)
        row = df.iloc[idx]
        results.append({
            "rank":                 None,
            "similarity":           round(float(score), 4),
            "match_type":           "semantic",
            "scheme_name":          row.get("scheme_name", ""),
            "scheme_name_hindi":    row.get("scheme_name_hindi", ""),
            "scheme_name_kannada":  row.get("scheme_name_kannada", ""),
            "tags":                 row.get("tags", ""),
            "level":                row.get("level", ""),
            "schemeCategory":       row.get("schemeCategory", ""),
            "slug":                 row.get("slug", ""),          # ← added
            "details_snippet":      safe_str(row.get("details", ""))[:250] + "...",
        })

    # ── Merge, sort, rank ─────────────────────────────────────────────────────
    result_df = pd.DataFrame(results)
    result_df = result_df.sort_values("similarity", ascending=False).head(top_k).reset_index(drop=True)
    result_df["rank"] = result_df.index + 1

    return result_df

print("✅ hybrid_search() ready with fuzzy and semantic matching")

✅ hybrid_search() ready with fuzzy and semantic matching


In [ ]:
def detect_language(text: str) -> str:
    """
    Detect language from script — no library needed.
    Kannada: Unicode U+0C80–U+0CFF
    Hindi:   Unicode U+0900–U+097F
    Else:    English
    """
    if re.search(r"[\u0C80-\u0CFF]", text):
        return "kn"
    elif re.search(r"[\u0900-\u097F]", text):
        return "hi"
    else:
        return "en"


def display_hybrid_results(query: str, top_k: int = TOP_K):
    print(f"\n🔍 Query: \"{query}\"")
    print("─" * 70)

    results_df = hybrid_search(query, top_k=top_k)

    if results_df.empty:
        print("No results found.")
        return

    for _, row in results_df.iterrows():
        print(f"\n#{int(row['rank'])}  [{row['similarity']:.4f}]  {row['match_type']}")
        print(f"  📌 {row['scheme_name']}")
        if row['scheme_name_hindi'] and str(row['scheme_name_hindi']) != 'nan':
            print(f"  📌 हिंदी  : {row['scheme_name_hindi']}")
        if row['scheme_name_kannada'] and str(row['scheme_name_kannada']) != 'nan':
            print(f"  📌 ಕನ್ನಡ  : {row['scheme_name_kannada']}")
        print(f"  🏷  Tags    : {row['tags']}")
        print(f"  🏛  Level   : {row['level']}   |   Category: {row['schemeCategory']}")
        if row['slug'] and str(row['slug']) != 'nan':
            print(f"  🔗 URL     : {BASE_URL}{row['slug']}")
        print(f"  📝 Details : {row['details_snippet']}")
        print("─" * 70)

    return results_df


print("✅ display_hybrid_results() ready")

✅ display_hybrid_results() ready


In [ ]:
import sys
if 'rapidfuzz' not in sys.modules:
    get_ipython().system('pip install -q rapidfuzz')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 51.6 MB/s eta 0:00:00


In [ ]:
from rapidfuzz import fuzz, process

BASE_URL = "https://www.myscheme.gov.in/schemes/"  # ← update this to your actual base URL

def safe_str(val):
    """Convert cell to string, empty string for NaN."""
    return "" if pd.isna(val) else str(val).strip()

def fuzzy_name_match(query: str, top_k: int = TOP_K) -> pd.DataFrame:
    """
    Fuzzy match query against scheme_name column.
    Catches misspellings like 'pradan' -> 'Pradhan'.
    """
    query_clean = query.strip().lower()

    scheme_names = df["scheme_name"].fillna("").tolist()

    matches = process.extract(
        query_clean,
        scheme_names,
        scorer=fuzz.token_sort_ratio,
        limit=top_k * 2
    )

    results = []
    seen = set()

    for match_text, score, idx in matches:
        if idx in seen or score < 40:
            continue
        seen.add(idx)
        row = df.iloc[idx]
        results.append({
            "idx":                  idx,
            "fuzzy_score":          score,
            "scheme_name":          row.get("scheme_name", ""),
            "scheme_name_hindi":    row.get("scheme_name_hindi", ""),
            "scheme_name_kannada":  row.get("scheme_name_kannada", ""),
            "tags":                 row.get("tags", ""),
            "level":                row.get("level", ""),
            "schemeCategory":       row.get("schemeCategory", ""),
            "slug":                 row.get("slug", ""),          # ← added
            "details_snippet":      safe_str(row.get("details", ""))[:250] + "...",
        })

    return pd.DataFrame(results)

In [ ]:
display_hybrid_results("pradan mantri awas yojana", top_k=5)


🔍 Query: "pradan mantri awas yojana"
──────────────────────────────────────────────────────────────────────

#1  [0.7273]  fuzzy (72.72727272727273%)
  📌 Pradhan Mantri Jan Dhan Yojana
  📌 हिंदी  : प्रधान मंत्री जान धन योजना
  📌 ಕನ್ನಡ  : ಪ್ರಧಾನ್ ಮಂತ್ರಿ ಜಾನ್ ಧನ್ ಯೋಜನಾ
  🏷  Tags    : Jan Dhan, Bank, Chota Khata, Insurance
  🏛  Level   : Central   |   Category: Banking,Financial Services and Insurance
  🔗 URL     : https://www.myscheme.gov.in/schemes/pmjdy
  📝 Details : A  National Mission for Financial Inclusion (NMFI), namely, Pradhan Mantri Jan Dhan Yojana (PMJDY) was launched across the Country  in August, 2014 by the Prime Minister with the idea to ensure that citizens envisage their financial activities. PMJDY...
──────────────────────────────────────────────────────────────────────

#2  [0.7241]  fuzzy (72.41379310344827%)
  📌 Pradhan Mantri Adarsh Gram Yojana
  📌 हिंदी  : प्रधान मंत्री आदर्श ग्राम योजना
  📌 ಕನ್ನಡ  : ಪ್ರಧಾನ್ ಮಂತ್ರಿ ಆದರ್ಶ್ ಗ್ರಾಂ ಯೋಜನಾ
  🏷  Tags    : Rural Developme

,rank,similarity,match_type,scheme_name,scheme_name_hindi,scheme_name_kannada,tags,level,schemeCategory,slug,details_snippet
0,1,0.7273,fuzzy (72.72727272727273%),Pradhan Mantri Jan Dhan Yojana,प्रधान मंत्री जान धन योजना,ಪ್ರಧಾನ್ ಮಂತ್ರಿ ಜಾನ್ ಧನ್ ಯೋಜನಾ,"Jan Dhan, Bank, Chota Khata, Insurance",Central,"Banking,Financial Services and Insurance",pmjdy,A National Mission for Financial Inclusion (N...
1,2,0.7241,fuzzy (72.41379310344827%),Pradhan Mantri Adarsh Gram Yojana,प्रधान मंत्री आदर्श ग्राम योजना,ಪ್ರಧಾನ್ ಮಂತ್ರಿ ಆದರ್ಶ್ ಗ್ರಾಂ ಯೋಜನಾ,"Rural Development, Social Justice",Central,"Agriculture,Rural & Environment",pmagy,With a view to enabling an area-based developm...
2,3,0.7213,fuzzy (72.1311475409836%),Pradhan Mantri Awaas Yojana - Gramin,प्रधान मंत्री आवास योजना - ग्रामीण,ಪ್ರಧಾನ್ ಮಂತ್ರಿ ಆವಾಸ್ ಯೋಜನಾ - ಗ್ರಾಮೀಣ್,"Rural Development, Housing, Loan, Financial As...",Central,Housing & Shelter,pmay-g,"Launch on 1st April 2016, Pradhan Mantri Awaas..."
3,4,0.7119,fuzzy (71.1864406779661%),Pradhan Mantri Awas Yojana - Urban,प्रधान मंत्री आवास योजना - अर्बन,ಪ್ರಧಾನ್ ಮಂತ್ರಿ ಆವಾಸ್ ಯೋಜನಾ - ಅರ್ಬನ್,"Housing, Urban, Rehabilitation, Loan, Sanitation",Central,"Housing & Shelter, Utility & Sanitation",pmay-u,"A flagship mission implemented by MoHUA, addre..."
4,5,0.6695,semantic,Saansad Adarsh Gram Yojana,सांसद आदर्श ग्राम योजना,ಸಾನ್ಸಡ್ ಆದರ್ಶ್ ಗ್ರಾಂ ಯೋಜನಾ,Rural Development,Central,"Agriculture,Rural & Environment",sagy,Introduction Saansad Adarsh Gram Yojana (SAGY)...


In [ ]:
user_query = input("Enter your query (English / Hindi / Kannada): ")
display_hybrid_results(user_query, top_k=TOP_K)

Enter your query (English / Hindi / Kannada): ಶೈಕ್ಷಣಿಕ ಯೋಜನೆ

🔍 Query: "ಶೈಕ್ಷಣಿಕ ಯೋಜನೆ"
──────────────────────────────────────────────────────────────────────

#1  [0.6311]  semantic
  📌 Prak Pariksha Prashikshan Kendra
  📌 हिंदी  : प्रक परीक्षा प्रशिक्षण केंद्र
  📌 ಕನ್ನಡ  : ಪ್ರಾಕ್ ಪರೀಕ್ಷಾ ಪ್ರಶಿಕ್ಷಣ ಕೇಂದ್ರ
  🏷  Tags    : Education, Learning, Coaching, Competitive Exams, Student
  🏛  Level   : State   |   Category: Education & Learning
  🔗 URL     : https://www.myscheme.gov.in/schemes/pppk
  📝 Details : The "Prak Pariksha Prashikshan Kendra" is an initiative launched by the Backward Classes and Extremely Backward Classes Welfare Department, Bihar, aimed at supporting underprivileged candidates aspiring to appear for competitive exams. It offers free...
──────────────────────────────────────────────────────────────────────

#2  [0.6182]  semantic
  📌 Dayanand Bandodkar Scheme For Higher Education For Orphans
  📌 हिंदी  : दयानन्द बंदोदकर स्कीम फॉर हायर एजुकेशन फॉर ऑर्फन्स
  📌 ಕನ್ನಡ  : ದಯಾ

,rank,similarity,match_type,scheme_name,scheme_name_hindi,scheme_name_kannada,tags,level,schemeCategory,slug,details_snippet
0,1,0.6311,semantic,Prak Pariksha Prashikshan Kendra,प्रक परीक्षा प्रशिक्षण केंद्र,ಪ್ರಾಕ್ ಪರೀಕ್ಷಾ ಪ್ರಶಿಕ್ಷಣ ಕೇಂದ್ರ,"Education, Learning, Coaching, Competitive Exa...",State,Education & Learning,pppk,"The ""Prak Pariksha Prashikshan Kendra"" is an i..."
1,2,0.6182,semantic,Dayanand Bandodkar Scheme For Higher Education...,दयानन्द बंदोदकर स्कीम फॉर हायर एजुकेशन फॉर ऑर्...,ದಯಾನಂದ್ ಬಾಂದೋಡ್ಕರ್ ಸ್ಕೀಮ್ ಫಾರ್ ಹೈಯರ್ ಎಜುಕೇಶನ್ ...,"Scholarship, Fee Waiver, Orphan, Higher Education",State,Education & Learning,dbsheo,"""Dayanand Bandodkar Scheme for Higher Educatio..."
2,3,0.6140,semantic,Dr. Ambedkar Medhavi Chattar Sansodhit Yojna,डॉ. आंबेडकर मेधावी छत्तर संसोधित योजना,ದ್ರ್. ಅಂಬೇಡ್ಕರ್ ಮೇಧಾವಿ ಚಟ್ಟರ್ ಸಂಶೋಧಿತ ಯೋಜನಾ,"Social Welfare, Financial Assistance,, The Sch...",State,Social welfare & Empowerment,amcsy,The Haryana government has prepared a scheme t...
3,4,0.6101,semantic,Mukhyamantri Konya Atmonirbhor Yojana,मुख्यमंत्री कोन्या आत्मोनिर्भर योजना,ಮುಖ್ಯಮಂತ್ರಿ ಕೋಣ್ಯಾ ಆತ್ಮೋನಿರ್ಬ್ಹೊರ್ ಯೋಜನಾ,"Scooty, Girl, Student, Degree",State,"Women and Child, Education & Learning",mkay,"The scheme ""Mukhyamantri Konya Atmonirbhor Yoj..."
4,5,0.5989,semantic,"""SUPER 30"" Scheme",सुपर ३० स्कीम,ಸೂಪರ್ ೩೦ ಸ್ಕೀಮ್,"School, Student, Education, Coaching, JEE, NEET",State,Education & Learning,s30s,"The scheme ""Super 30"" was launched by the Dire..."


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# TOPIC-BASED RETRIEVAL ACCURACY — English + Hindi + Kannada
# ─────────────────────────────────────────────────────────────────────────────

from rapidfuzz import fuzz

def result_is_relevant(row: pd.Series, relevance_keywords: list[str]) -> bool:
    """
    Checks relevance across ALL language fields — so a Hindi query can match
    against English tags and vice versa (since your search_text merges all).
    """
    haystack = " ".join([
        str(row.get("scheme_name", "")),
        str(row.get("scheme_name_hindi", "")),
        str(row.get("scheme_name_kannada", "")),
        str(row.get("tags", "")),
        str(row.get("tags_hi", "")),
        str(row.get("tags_kn", "")),
        str(row.get("schemeCategory", "")),
    ]).lower()

    return any(kw.lower() in haystack for kw in relevance_keywords)


def evaluate_text_retrieval(test_cases: list[dict], top_k: int = TOP_K) -> dict:
    """
    Topic-aware retrieval accuracy — works for English, Hindi, and Kannada queries.

    Each test case:
        'query'              — user query in any language
        'lang'               — 'en', 'hi', or 'kn' (for display only)
        'relevance_keywords' — keywords in ANY language; a result is a HIT if
                               ANY keyword appears across all language fields
        'min_hits'           — (optional) how many of top_k must be relevant. Default = 1
    """
    total_queries    = len(test_cases)
    query_hits       = 0
    all_precisions   = []
    reciprocal_ranks = []
    breakdown        = []

    for case in test_cases:
        query    = case["query"]
        keywords = case["relevance_keywords"]
        lang     = case.get("lang", "en")
        min_hits = case.get("min_hits", 1)

        try:
            results_df = hybrid_search(query, top_k=top_k)
        except Exception as e:
            print(f"⚠️  Search failed for '{query}': {e}")
            breakdown.append({"query": query, "lang": lang, "error": str(e)})
            all_precisions.append(0)
            reciprocal_ranks.append(0)
            continue

        relevance_flags = [
            result_is_relevant(row, keywords)
            for _, row in results_df.iterrows()
        ]

        relevant_count = sum(relevance_flags)
        precision      = relevant_count / len(results_df) if results_df.shape[0] > 0 else 0
        first_hit_rank = next((i + 1 for i, f in enumerate(relevance_flags) if f), None)
        rr             = (1 / first_hit_rank) if first_hit_rank else 0
        passed         = relevant_count >= min_hits

        if passed:
            query_hits += 1

        all_precisions.append(precision)
        reciprocal_ranks.append(rr)

        breakdown.append({
            "query":          query,
            "lang":           lang,
            "keywords":       keywords,
            "relevant_found": relevant_count,
            "precision":      round(precision, 3),
            "first_hit_rank": first_hit_rank,
            "passed":         passed,
            "top_results":    results_df["scheme_name"].tolist() if not results_df.empty else [],
        })

    hit_rate  = query_hits / total_queries if total_queries > 0 else 0
    mean_prec = sum(all_precisions) / total_queries if total_queries > 0 else 0
    mrr       = sum(reciprocal_ranks) / total_queries if total_queries > 0 else 0

    # ── Overall summary ───────────────────────────────────────────────────────
    print(f"\n{'='*62}")
    print(f"  TEXT RETRIEVAL ACCURACY  (top_k = {top_k})")
    print(f"{'='*62}")
    print(f"  Queries evaluated  : {total_queries}")
    print(f"  Queries passed     : {query_hits}")
    print(f"  Hit Rate @{top_k:<2}       : {hit_rate:.1%}")
    print(f"  Mean Precision @{top_k:<2} : {mean_prec:.1%}")
    print(f"  MRR                : {mrr:.4f}")
    print(f"{'='*62}")

    # ── Per-language breakdown ────────────────────────────────────────────────
    for lang_code, label in [("en", "English 🇬🇧"), ("hi", "Hindi 🇮🇳"), ("kn", "Kannada 🌿")]:
        lang_cases = [b for b in breakdown if b.get("lang") == lang_code and "error" not in b]
        if not lang_cases:
            continue
        lang_hit  = sum(1 for b in lang_cases if b["passed"]) / len(lang_cases)
        lang_prec = sum(b["precision"] for b in lang_cases) / len(lang_cases)
        print(f"\n  {label}  ({len(lang_cases)} queries)")
        print(f"    Hit Rate @{top_k}: {lang_hit:.1%}   |   Mean Precision: {lang_prec:.1%}")

    # ── Per-query detail ──────────────────────────────────────────────────────
    print(f"\n\n{'Lang':<5} {'Query':<38} {'Pass':>5} {'Rel':>4} {'P@k':>6} {'Rank':>6}")
    print("─" * 65)
    for b in breakdown:
        if "error" in b:
            print(f"  {'ERR':<5} {b['query'][:38]}")
            continue
        flag     = "✅" if b["passed"] else "❌"
        rank_str = str(b["first_hit_rank"]) if b["first_hit_rank"] else "—"
        lang_tag = {"en": "EN", "hi": "HI", "kn": "KN"}.get(b["lang"], "??")
        print(
            f"  {lang_tag:<5} {b['query'][:38]:<38} {flag:>5}"
            f" {b['relevant_found']:>4} {b['precision']:>6.1%} {rank_str:>6}"
        )
        if not b["passed"]:
            for name in b["top_results"][:2]:
                print(f"        ↳ got: {name[:60]}")

    return {
        "hit_rate": hit_rate, "mean_precision": mean_prec,
        "mrr": mrr, "passed": query_hits, "total": total_queries,
        "breakdown": breakdown,
    }


# ─────────────────────────────────────────────────────────────────────────────
# TEST CASES — English + Hindi + Kannada
# Keywords span all 3 languages so a match in ANY field counts as relevant
# ─────────────────────────────────────────────────────────────────────────────

test_cases = [

    # ── English ──────────────────────────────────────────────────────────────
    {
        "query": "health insurance for construction workers",
        "lang": "en",
        "relevance_keywords": ["construction", "health", "insurance", "worker", "medical",
                               "निर्माण", "स्वास्थ्य", "ನಿರ್ಮಾಣ", "ಆರೋಗ್ಯ"],
    },
    {
        "query": "scholarship for poor students",
        "lang": "en",
        "relevance_keywords": ["scholarship", "student", "education", "BPL", "financial assistance",
                               "छात्रवृत्ति", "विद्यार्थी", "ವಿದ್ಯಾರ್ಥಿ", "ಶಿಕ್ಷಣ"],
    },
    {
        "query": "housing scheme for rural families",
        "lang": "en",
        "relevance_keywords": ["housing", "rural", "awas", "shelter", "construction",
                               "आवास", "ग्रामीण", "ಆವಾಸ", "ಗ್ರಾಮೀಣ"],
    },
    {
        "query": "women self employment loan",
        "lang": "en",
        "relevance_keywords": ["women", "loan", "self employment", "entrepreneurship",
                               "महिला", "ऋण", "ಮಹಿಳೆ", "ಸಾಲ"],
    },
    {
        "query": "old age pension",
        "lang": "en",
        "relevance_keywords": ["pension", "elderly", "old age", "senior",
                               "पेंशन", "वृद्ध", "ಪಿಂಚಣಿ", "ವೃದ್ಧ"],
    },
    {
        "query": "disability financial assistance",
        "lang": "en",
        "relevance_keywords": ["disability", "disabled", "PwD", "persons with disabilities",
                               "विकलांग", "ಅಂಗವಿಕಲ", "ದಿವ್ಯಾಂಗ"],
    },
    {
        "query": "skill training for unemployed youth",
        "lang": "en",
        "relevance_keywords": ["skill", "training", "employment", "youth", "vocational",
                               "कौशल", "प्रशिक्षण", "ತರಬೇತಿ", "ಕೌಶಲ"],
    },

    # ── Hindi ─────────────────────────────────────────────────────────────────
    {
        "query": "निर्माण मजदूरों के लिए स्वास्थ्य बीमा",   # health insurance for construction workers
        "lang": "hi",
        "relevance_keywords": ["construction", "health", "insurance", "worker", "medical",
                               "निर्माण", "स्वास्थ्य", "बीमा", "मजदूर", "श्रमिक"],
    },
    {
        "query": "गरीब छात्रों के लिए छात्रवृत्ति",          # scholarship for poor students
        "lang": "hi",
        "relevance_keywords": ["scholarship", "student", "education", "BPL",
                               "छात्रवृत्ति", "विद्यार्थी", "शिक्षा", "गरीब"],
    },
    {
        "query": "महिला स्वरोजगार ऋण योजना",                 # women self employment loan
        "lang": "hi",
        "relevance_keywords": ["women", "loan", "entrepreneurship", "self employment",
                               "महिला", "ऋण", "स्वरोजगार", "उद्यम"],
    },
    {
        "query": "वृद्धावस्था पेंशन",                        # old age pension
        "lang": "hi",
        "relevance_keywords": ["pension", "elderly", "old age",
                               "पेंशन", "वृद्ध", "बुजुर्ग"],
    },
    {
        "query": "ग्रामीण आवास योजना",                       # rural housing scheme
        "lang": "hi",
        "relevance_keywords": ["housing", "rural", "awas",
                               "आवास", "ग्रामीण", "निर्माण"],
    },
    {
        "query": "अनुसूचित जाति छात्र वित्तीय सहायता",       # SC student financial help
        "lang": "hi",
        "relevance_keywords": ["SC", "scheduled caste", "student", "financial assistance",
                               "अनुसूचित जाति", "विद्यार्थी", "आर्थिक सहायता"],
    },

    # ── Kannada ───────────────────────────────────────────────────────────────
    {
        "query": "ನಿರ್ಮಾಣ ಕಾರ್ಮಿಕರಿಗೆ ಆರೋಗ್ಯ ವಿಮೆ",          # health insurance for construction workers
        "lang": "kn",
        "relevance_keywords": ["construction", "health", "insurance", "worker",
                               "ನಿರ್ಮಾಣ", "ಆರೋಗ್ಯ", "ವಿಮೆ", "ಕಾರ್ಮಿಕ"],
    },
    {
        "query": "ಬಡ ವಿದ್ಯಾರ್ಥಿಗಳಿಗೆ ವಿದ್ಯಾರ್ಥಿವೇತನ",         # scholarship for poor students
        "lang": "kn",
        "relevance_keywords": ["scholarship", "student", "education",
                               "ವಿದ್ಯಾರ್ಥಿವೇತನ", "ವಿದ್ಯಾರ್ಥಿ", "ಶಿಕ್ಷಣ"],
    },
    {
        "query": "ಮಹಿಳೆಯರಿಗೆ ಸ್ವಯಂ ಉದ್ಯೋಗ ಸಾಲ",               # women self employment loan
        "lang": "kn",
        "relevance_keywords": ["women", "loan", "entrepreneurship",
                               "ಮಹಿಳೆ", "ಸಾಲ", "ಉದ್ಯೋಗ", "ಸಬಲೀಕರಣ"],
    },
    {
        "query": "ವೃದ್ಧಾಪ್ಯ ಪಿಂಚಣಿ ಯೋಜನೆ",                    # old age pension
        "lang": "kn",
        "relevance_keywords": ["pension", "elderly",
                               "ಪಿಂಚಣಿ", "ವೃದ್ಧ", "ಹಿರಿಯ ನಾಗರಿಕ"],
    },
    {
        "query": "ಗ್ರಾಮೀಣ ವಸತಿ ಯೋಜನೆ",                        # rural housing scheme
        "lang": "kn",
        "relevance_keywords": ["housing", "rural", "awas",
                               "ಆವಾಸ", "ಗ್ರಾಮೀಣ", "ವಸತಿ", "ನಿರ್ಮಾಣ"],
    },
    {
        "query": "ಕೌಶಲ್ಯ ತರಬೇತಿ ಯೋಜನೆ",                       # skill training scheme
        "lang": "kn",
        "relevance_keywords": ["skill", "training", "employment", "vocational",
                               "ಕೌಶಲ", "ತರಬೇತಿ", "ಉದ್ಯೋಗ"],
    },
]

# ─────────────────────────────────────────────────────────────────────────────
# RUN
# ─────────────────────────────────────────────────────────────────────────────

results = evaluate_text_retrieval(test_cases, top_k=TOP_K)


  TEXT RETRIEVAL ACCURACY  (top_k = 5)
  Queries evaluated  : 19
  Queries passed     : 18
  Hit Rate @5        : 94.7%
  Mean Precision @5  : 86.3%
  MRR                : 0.9474

  English 🇬🇧  (7 queries)
    Hit Rate @5: 100.0%   |   Mean Precision: 94.3%

  Hindi 🇮🇳  (6 queries)
    Hit Rate @5: 100.0%   |   Mean Precision: 96.7%

  Kannada 🌿  (6 queries)
    Hit Rate @5: 83.3%   |   Mean Precision: 66.7%


Lang  Query                                   Pass  Rel    P@k   Rank
─────────────────────────────────────────────────────────────────
  EN    health insurance for construction work     ✅    5 100.0%      1
  EN    scholarship for poor students              ✅    5 100.0%      1
  EN    housing scheme for rural families          ✅    4  80.0%      1
  EN    women self employment loan                 ✅    4  80.0%      1
  EN    old age pension                            ✅    5 100.0%      1
  EN    disability financial assistance            ✅    5 100.0%      1
  EN    skill tra

In [ ]:
!pip install -q openai-whisper

In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────────
!pip install -q openai-whisper pydub ffmpeg-python
!apt-get install -qq -y ffmpeg

In [ ]:
# ── Cell 2: Imports ───────────────────────────────────────────────────────────
import whisper
import warnings
import subprocess
import os
warnings.filterwarnings("ignore")

In [ ]:
# ── Cell 3: Load Whisper — use 'medium' for much better Hindi/Kannada accuracy
print("Loading Whisper model (medium)...")
stt_model = whisper.load_model("medium")  # 'small' misses Kannada; 'medium' is the sweet spot
print("✅ STT model loaded")

Loading Whisper model (medium)...


100%|██████████████████████████████████████| 1.42G/1.42G [00:06<00:00, 233MiB/s]


✅ STT model loaded


In [ ]:
# ── Cell 4: Audio pre-processing + robust STT ─────────────────────────────────

def convert_to_wav(input_path: str, output_path: str = "query_converted.wav") -> str:
    """
    Convert any audio (webm, ogg, mp3, etc.) to 16kHz mono WAV.
    Whisper expects 16kHz mono — skipping this causes garbled transcriptions.
    """
    cmd = [
        "ffmpeg", "-y",
        "-i", input_path,
        "-ar", "16000",    # 16 kHz sample rate
        "-ac", "1",        # mono channel
        "-c:a", "pcm_s16le",  # 16-bit PCM
        output_path
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"ffmpeg conversion failed:\n{result.stderr}")
    return output_path


# Language code mapping for display
LANG_DISPLAY = {
    "hi": "Hindi / हिंदी",
    "kn": "Kannada / ಕನ್ನಡ",
    "en": "English",
    "te": "Telugu",
    "ta": "Tamil",
    "mr": "Marathi",
}


def transcribe_audio(audio_path: str, language: str = None) -> tuple[str, str]:
    """
    Transcribe audio to text using Whisper.

    Args:
        audio_path: Path to the audio file.
        language:   ISO 639-1 code to force language ('hi', 'kn', 'en').
                    Pass None to let Whisper auto-detect.

    Returns:
        (transcribed_text, detected_language_code)

    Key fix: always convert to 16kHz WAV first, and pass language explicitly
    when you know it — this eliminates the #1 cause of bad transcriptions.
    """
    # Step 1: Convert to proper WAV format
    wav_path = convert_to_wav(audio_path)

    # Step 2: Build transcription options
    options = {
        "task": "transcribe",      # keep original language (do NOT use 'translate')
        "beam_size": 5,            # more accurate than greedy (beam_size=1)
        "best_of": 5,
        "temperature": 0.0,        # deterministic output
        "condition_on_previous_text": False,  # prevents hallucination loops
        "fp16": False,             # safer on CPU; set True if on GPU
    }

    if language:
        options["language"] = language  # e.g. 'hi' or 'kn'

    # Step 3: Transcribe
    result = stt_model.transcribe(wav_path, **options)
    text = result["text"].strip()
    detected = result.get("language", "unknown")

    # Clean up temp file
    if os.path.exists(wav_path):
        os.remove(wav_path)

    return text, detected


def search_schemes_by_voice(audio_path: str, language: str = None, top_k: int = TOP_K):
    """
    Full pipeline: audio file → Whisper STT → hybrid_search → display results.

    Args:
        audio_path: Path to recorded audio.
        language:   Force language for better accuracy:
                      'hi' = Hindi, 'kn' = Kannada, 'en' = English, None = auto-detect
        top_k:      Number of results to return.
    """
    print(f"🎙️  Transcribing audio...")

    try:
        transcribed_text, detected_lang = transcribe_audio(audio_path, language=language)
    except Exception as e:
        print(f"❌ Transcription failed: {e}")
        return

    lang_label = LANG_DISPLAY.get(detected_lang, detected_lang)
    print(f"✅ Detected language : {lang_label}")
    print(f"✅ Transcribed text  : \"{transcribed_text}\"")

    if not transcribed_text:
        print("⚠️ Empty transcription. Please speak clearly and try again.")
        return

    return display_hybrid_results(transcribed_text, top_k=top_k)

In [ ]:
# ── Cell 5: Browser microphone recorder + language selector ───────────────────
from google.colab import output as colab_output
from base64 import b64decode
from IPython.display import display, Javascript, HTML
import ipywidgets as widgets

# ── Language selector widget ──────────────────────────────────────────────────
lang_selector = widgets.ToggleButtons(
    options=[
        ("Auto-detect", None),
        ("Hindi / हिंदी", "hi"),
        ("Kannada / ಕನ್ನಡ", "kn"),
        ("English", "en"),
    ],
    description="Language:",
    button_style="info",
    value=None,
)
display(lang_selector)

def record_and_search(filename: str = "input_query.webm"):
    """Record from browser mic, save, and run voice search."""

    js = Javascript("""
    async function recordAudio() {
      // ── UI ──────────────────────────────────────────────────────────────
      const div = document.createElement('div');
      div.style.cssText = 'padding:12px;background:#f0f4ff;border-radius:8px;margin:8px 0;font-family:sans-serif;';

      const status = document.createElement('p');
      status.textContent = '🔴 Recording... Speak your query now';
      status.style.color = '#c0392b';

      const button = document.createElement('button');
      button.textContent = '⏹ Stop Recording';
      button.style.cssText = 'padding:8px 16px;background:#e74c3c;color:white;border:none;border-radius:4px;cursor:pointer;font-size:14px;';

      div.appendChild(status);
      div.appendChild(button);
      document.body.appendChild(div);

      // ── Recording ───────────────────────────────────────────────────────
      const stream = await navigator.mediaDevices.getUserMedia({
        audio: {
          channelCount: 1,        // mono
          sampleRate: 16000,      // match Whisper's expected rate
          echoCancellation: true,
          noiseSuppression: true, // cleaner audio = better STT
          autoGainControl: true,
        }
      });

      // Use opus/webm (best for speech), fallback to default
      const mimeType = MediaRecorder.isTypeSupported('audio/webm;codecs=opus')
                       ? 'audio/webm;codecs=opus'
                       : 'audio/webm';

      const recorder = new MediaRecorder(stream, { mimeType });
      const chunks = [];

      recorder.ondataavailable = e => { if (e.data.size > 0) chunks.push(e.data); };
      recorder.start(100); // collect chunks every 100ms

      return new Promise(resolve => {
        recorder.onstop = () => {
          stream.getTracks().forEach(t => t.stop());
          status.textContent = '✅ Recording complete. Processing...';
          button.disabled = true;

          const blob = new Blob(chunks, { type: mimeType });
          const reader = new FileReader();
          reader.onloadend = () => resolve(reader.result.split(',')[1]);
          reader.readAsDataURL(blob);
        };

        button.onclick = () => recorder.stop();
      });
    }
    """)
    display(js)

    data = colab_output.eval_js("recordAudio()")
    with open(filename, "wb") as f:
        f.write(b64decode(data))

    size = os.path.getsize(filename)
    print(f"📁 Saved audio: {filename} ({size:,} bytes)")

    if size < 1000:
        print("⚠️ Audio file too small — microphone may not have captured anything.")
        print("   Make sure you grant mic permission and speak for at least 1–2 seconds.")
        return

    # Use the selected language from the widget
    chosen_lang = lang_selector.value
    print(f"🌐 Using language hint: {chosen_lang or 'auto-detect'}")

    search_schemes_by_voice(filename, language=chosen_lang)


# ── Run ───────────────────────────────────────────────────────────────────────
print("Select your language above ↑, then run the cell below to record.")

ToggleButtons(button_style='info', description='Language:', options=(('Auto-detect', None), ('Hindi / हिंदी', …

Select your language above ↑, then run the cell below to record.


In [ ]:
# ── Cell 6: Record and search ─────────────────────────────────────────────────
# Run this cell each time you want to do a voice search.
record_and_search()

<IPython.core.display.Javascript object>

📁 Saved audio: input_query.webm (64,425 bytes)
🌐 Using language hint: en
🎙️  Transcribing audio...
✅ Detected language : English
✅ Transcribed text  : "Education scheme."

🔍 Query: "Education scheme."
──────────────────────────────────────────────────────────────────────

#1  [0.7368]  fuzzy (73.6842105263158%)
  📌 Education Loan Scheme
  📌 हिंदी  : एजुकेशन लोन स्कीम
  📌 ಕನ್ನಡ  : ಎಜುಕೇಶನ್ ಲೋನ್ ಸ್ಕೀಮ್
  🏷  Tags    : Higher Education, Loans To Students, National Scheduled Castes Finance And Development Corporation
  🏛  Level   : Central   |   Category: Education & Learning, Banking,Financial Services and Insurance
  🔗 URL     : https://www.myscheme.gov.in/schemes/els
  📝 Details : A scheme named " Education Loan Scheme " by National Scheduled Castes Finance and Development Corporation (NSFDC) under the M/o Social Justice and Empowerment provides loans to students from Scheduled Castes who are pursuing full-time professional or...
──────────────────────────────────────────────────────────